In [14]:
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

OUTPUT_FILE = 'p1_alerts.csv'

SEVERITY_ORDER = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}


def resolve_data_dir(user_path: str | None = None) -> Path:
    candidates = []
    if user_path:
        candidates.append(Path(user_path))
    candidates += [
        Path('data_dir/equity'),
        Path('E-Hackathon-main/E-Hackathon/src/data_dir/equity'),
        Path('E-Hackathon-main/E-Hackathon/data_dir/equity'),
        Path('/mnt/data/ehack/E-Hackathon-main/E-Hackathon/src/data_dir/equity'),
        Path('/mnt/data/ehack/E-Hackathon-main/E-Hackathon/data_dir/equity'),
    ]
    for c in candidates:
        if (c / 'market_data.csv').exists():
            return c
    raise FileNotFoundError('Could not locate equity data_dir with market_data.csv, ohlcv.csv, trade_data.csv')


def load_data(data_dir: str | None = None):
    base = resolve_data_dir(data_dir)
    mkt = pd.read_csv(base / 'market_data.csv', parse_dates=['timestamp'])
    ohlcv = pd.read_csv(base / 'ohlcv.csv', parse_dates=['trade_date'])
    trades = pd.read_csv(base / 'trade_data.csv', parse_dates=['timestamp'])
    return base, mkt, ohlcv, trades


def rolling_zscore(s: pd.Series, window: int = 60, min_periods: int = 20) -> pd.Series:
    mean = s.rolling(window, min_periods=min_periods).mean()
    std = s.rolling(window, min_periods=min_periods).std().replace(0, np.nan)
    return (s - mean) / std


def compute_market_features(mkt: pd.DataFrame) -> pd.DataFrame:
    df = mkt.copy().sort_values(['sec_id', 'timestamp']).reset_index(drop=True)
    bid_sizes = [f'bid_size_level{i:02d}' for i in range(1, 11)]
    ask_sizes = [f'ask_size_level{i:02d}' for i in range(1, 11)]
    bid_prices = [f'bid_price_level{i:02d}' for i in range(1, 11)]
    ask_prices = [f'ask_price_level{i:02d}' for i in range(1, 11)]
    for c in bid_sizes + ask_sizes + bid_prices + ask_prices:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    df['total_bid'] = df[bid_sizes].sum(axis=1)
    df['total_ask'] = df[ask_sizes].sum(axis=1)
    denom = (df['total_bid'] + df['total_ask']).replace(0, np.nan)
    df['obi'] = (df['total_bid'] - df['total_ask']) / denom
    df['spread_bps'] = (df['ask_price_level01'] - df['bid_price_level01']) / df['bid_price_level01'].replace(0, np.nan) * 1e4
    df['depth_ratio'] = df['bid_size_level01'] / df['ask_size_level01'].replace(0, np.nan)
    df['bid_conc'] = df['bid_size_level01'] / df['total_bid'].replace(0, np.nan)
    df['ask_conc'] = df['ask_size_level01'] / df['total_ask'].replace(0, np.nan)
    df['l1_share'] = df[['bid_size_level01', 'ask_size_level01']].sum(axis=1) / denom
    df['weighted_mid'] = (
        df['bid_price_level01'] * df['ask_size_level01'] + df['ask_price_level01'] * df['bid_size_level01']
    ) / (df['bid_size_level01'] + df['ask_size_level01']).replace(0, np.nan)
    df['mid_return_bps_1m'] = df.groupby('sec_id')['weighted_mid'].pct_change() * 1e4

    for feat in ['obi', 'spread_bps', 'depth_ratio', 'bid_conc', 'ask_conc', 'l1_share', 'mid_return_bps_1m']:
        df[f'{feat}_z'] = df.groupby('sec_id')[feat].transform(lambda x: rolling_zscore(x))

    df['trade_date'] = df['timestamp'].dt.strftime('%Y-%m-%d')
    df['time_window_start'] = df['timestamp'].dt.strftime('%H:%M:%S')
    df['minute_of_day'] = df['timestamp'].dt.hour * 60 + df['timestamp'].dt.minute
    df['seq_in_day'] = df.groupby(['sec_id', 'trade_date']).cumcount()
    df['is_open_noise'] = df['seq_in_day'] < 10
    return df


def compute_trade_features(trades: pd.DataFrame) -> pd.DataFrame:
    tr = trades.copy().sort_values(['sec_id', 'timestamp']).reset_index(drop=True)
    tr['notional'] = pd.to_numeric(tr['price'], errors='coerce') * pd.to_numeric(tr['quantity'], errors='coerce')
    tr['trade_date'] = tr['timestamp'].dt.strftime('%Y-%m-%d')
    tr['minute_bucket'] = tr['timestamp'].dt.floor('min')
    return tr


def make_runs(df: pd.DataFrame, time_col: str = 'timestamp', max_gap='2min') -> pd.Series:
    gap = pd.Timedelta(max_gap)
    return df.groupby(['sec_id', 'trade_date'])[time_col].transform(lambda x: (x.diff().gt(gap) | x.diff().isna()).cumsum())


def detect_imbalance_runs(mkt: pd.DataFrame) -> list[dict]:
    strong = mkt[(~mkt['is_open_noise']) & (mkt['obi'].abs() >= 0.78) & (mkt['obi_z'].abs() >= 2.5)].copy()
    if strong.empty:
        return []
    strong['run_id'] = make_runs(strong)
    alerts = []
    for (sec_id, trade_date, run_id), grp in strong.groupby(['sec_id', 'trade_date', 'run_id']):
        duration = len(grp)
        if duration < 4:
            continue
        score = grp['obi_z'].abs().max() + grp['l1_share_z'].abs().fillna(0).max() + duration / 3
        side = 'bid' if grp['obi'].mean() > 0 else 'ask'
        max_obi = grp['obi'].abs().max()
        severity = 'HIGH' if (max_obi > 0.92 or duration >= 8) else 'MEDIUM'
        alerts.append({
            'sec_id': sec_id,
            'trade_date': trade_date,
            'timestamp': grp['timestamp'].min(),
            'time_window_start': grp['timestamp'].min().strftime('%H:%M:%S'),
            'anomaly_type': 'sustained_order_book_imbalance',
            'severity': severity,
            'score': round(score, 3),
            'remarks': (
                f'Sustained {side}-side imbalance for {duration} consecutive minutes; '
                f'max |OBI|={max_obi:.3f}, OBI z-score peaked at {grp["obi_z"].abs().max():.1f}σ. '
                f'Top-of-book concentration was elevated versus the ticker baseline, suggesting a persistent '
                f'book skew rather than a one-minute spike.'
            )
        })
    return alerts


def detect_spread_depth_episodes(mkt: pd.DataFrame) -> list[dict]:
    mask = (
        (mkt['spread_bps_z'].abs() >= 3.5) &
        ((mkt['depth_ratio_z'].abs() >= 3.0) | (mkt['bid_conc'].ge(0.70)) | (mkt['ask_conc'].ge(0.70)))
    )
    strong = mkt[mask & (~mkt['is_open_noise'])].copy()
    if strong.empty:
        return []
    strong['run_id'] = make_runs(strong, max_gap='3min')
    alerts = []
    for (sec_id, trade_date, run_id), grp in strong.groupby(['sec_id', 'trade_date', 'run_id']):
        if len(grp) < 2:
            continue
        bid_heavy = grp['depth_ratio'].median() > 1
        score = grp['spread_bps_z'].abs().max() + grp['depth_ratio_z'].abs().fillna(0).max() + len(grp)
        severity = 'HIGH' if score >= 11 else 'MEDIUM'
        alerts.append({
            'sec_id': sec_id,
            'trade_date': trade_date,
            'timestamp': grp['timestamp'].min(),
            'time_window_start': grp['timestamp'].min().strftime('%H:%M:%S'),
            'anomaly_type': 'spread_depth_dislocation',
            'severity': severity,
            'score': round(score, 3),
            'remarks': (
                f'Spread widened abnormally for {len(grp)} minutes with max spread z-score '
                f'{grp["spread_bps_z"].abs().max():.1f}σ and top-of-book depth skewed toward the '
                f'{"bid" if bid_heavy else "ask"} side. This combination is more suspicious than a standalone '
                f'spread spike because quote width and displayed depth moved together.'
            )
        })
    return alerts


def detect_cancel_bursts(trades: pd.DataFrame, mkt: pd.DataFrame) -> list[dict]:
    tr = trades.copy()
    canc = tr[(tr['order_status'].eq('CANCELLED')) & (tr['order_type'].eq('LIMIT'))].copy()
    if canc.empty:
        return []
    fills = tr[tr['order_status'].eq('FILLED')].copy()

    # dynamic size threshold by sec_id using cancelled-notional distribution
    sec_threshold = canc.groupby('sec_id')['notional'].quantile(0.90).rename('sec_notional_q90')
    canc = canc.merge(sec_threshold, on='sec_id', how='left')
    canc = canc[canc['notional'] >= canc['sec_notional_q90'].fillna(canc['notional'].median())]
    if canc.empty:
        return []

    canc['bucket_15m'] = canc['timestamp'].dt.floor('15min')
    alerts = []
    grp_cols = ['trader_id', 'sec_id', 'trade_date', 'bucket_15m', 'side']
    for keys, grp in canc.groupby(grp_cols):
        trader_id, sec_id, trade_date, bucket_15m, side = keys
        n_cancel = len(grp)
        if n_cancel < 4:
            continue
        full_grp = tr[(tr['trader_id'] == trader_id) & (tr['sec_id'] == sec_id) & (tr['trade_date'] == trade_date) & (tr['timestamp'].between(bucket_15m, bucket_15m + pd.Timedelta('15min')))]
        cancel_ratio = (full_grp['order_status'] == 'CANCELLED').mean() if len(full_grp) else 1.0
        if cancel_ratio < 0.80:
            continue
        nearby_fill = fills[(fills['sec_id'] == sec_id) & (fills['trade_date'] == trade_date) & (fills['timestamp'].between(bucket_15m - pd.Timedelta('5min'), bucket_15m + pd.Timedelta('20min')))]
        opposite_side = 'SELL' if side == 'BUY' else 'BUY'
        opp_fills = nearby_fill[nearby_fill['side'] == opposite_side]
        mkt_slice = mkt[(mkt['sec_id'] == sec_id) & (mkt['trade_date'] == trade_date) & (mkt['timestamp'].between(bucket_15m - pd.Timedelta('2min'), bucket_15m + pd.Timedelta('17min')))]
        supportive = False
        if not mkt_slice.empty:
            supportive = ((mkt_slice['obi'].abs().max() >= 0.75) or (mkt_slice['depth_ratio_z'].abs().max() >= 3))
        if not supportive and opp_fills.empty:
            continue
        total_notional = grp['notional'].sum()
        score = n_cancel + 3 * cancel_ratio + np.log1p(total_notional / 1000) + (2 if supportive else 0)
        severity = 'HIGH' if (n_cancel >= 6 and supportive) else 'MEDIUM'
        remarks = (
            f'Trader {trader_id} placed {n_cancel} unusually large {side} LIMIT orders that were cancelled '
            f'within a 15-minute window; cancel ratio for the trader-window was {cancel_ratio:.0%}. '
            f'Total cancelled notional was ${total_notional:,.0f}'
        )
        if supportive:
            remarks += f', and the order book showed supporting imbalance/depth distortion during the same episode.'
        if not opp_fills.empty:
            remarks += f' Opposite-side filled trades appeared nearby, which strengthens the spoofing interpretation.'
        alerts.append({
            'sec_id': sec_id,
            'trade_date': trade_date,
            'timestamp': bucket_15m,
            'time_window_start': bucket_15m.strftime('%H:%M:%S'),
            'anomaly_type': 'spoofing_cancel_burst',
            'severity': severity,
            'score': round(score, 3),
            'remarks': remarks + ''
        })
    return alerts


def detect_marking_close(trades: pd.DataFrame, ohlcv: pd.DataFrame) -> list[dict]:
    tr = trades[trades['order_status'].eq('FILLED')].copy()
    tr['hour'] = tr['timestamp'].dt.hour
    tr['minute'] = tr['timestamp'].dt.minute
    close = tr[(tr['hour'] == 15) & (tr['minute'] >= 50)].copy()
    if close.empty:
        return []
    daily = tr.groupby(['sec_id', 'trade_date'])['quantity'].sum().rename('day_qty')
    csum = close.groupby(['sec_id', 'trade_date'])['quantity'].sum().rename('close_qty')
    summary = pd.concat([daily, csum], axis=1).fillna(0).reset_index()
    summary['close_share'] = summary['close_qty'] / summary['day_qty'].replace(0, np.nan)
    summary = summary[summary['close_share'] >= 0.55]

    # compare close price move vs normal minute move for the day
    ohl = ohlcv.copy()
    ohl['trade_date'] = ohl['trade_date'].dt.strftime('%Y-%m-%d')
    alerts = []
    for _, row in summary.iterrows():
        sec_id, trade_date = row['sec_id'], row['trade_date']
        close_slice = close[(close['sec_id'] == sec_id) & (close['trade_date'] == trade_date)]
        if close_slice.empty:
            continue
        largest = close_slice['quantity'].max()
        score = 8 * row['close_share'] + np.log1p(largest)
        severity = 'HIGH' if row['close_share'] >= 0.75 else 'MEDIUM'
        alerts.append({
            'sec_id': sec_id,
            'trade_date': trade_date,
            'timestamp': close_slice['timestamp'].min(),
            'time_window_start': close_slice['timestamp'].min().strftime('%H:%M:%S'),
            'anomaly_type': 'marking_the_close',
            'severity': severity,
            'score': round(score, 3),
            'remarks': (
                f'Filled volume in the final 10 minutes accounted for {row["close_share"]:.1%} of the day\'s '
                f'filled quantity. Largest close-window order was {largest:,} shares, making the end-of-session '
                f'activity unusually concentrated.'
            )
        })
    return alerts


def detect_dbscan_windows(mkt: pd.DataFrame) -> list[dict]:
    alerts = []
    feats = ['obi_z', 'spread_bps_z', 'depth_ratio_z', 'l1_share_z']
    for sec_id, grp in mkt.groupby('sec_id'):
        sub = grp.dropna(subset=feats).copy()
        if len(sub) < 40:
            continue
        X = StandardScaler().fit_transform(sub[feats])
        labels = DBSCAN(eps=0.9, min_samples=6).fit_predict(X)
        sub['db_label'] = labels
        ano = sub[(sub['db_label'] == -1) & (~sub['is_open_noise'])].copy()
        if ano.empty:
            continue
        # keep only outliers with another strong signal to avoid standalone noise
        ano = ano[(ano['obi_z'].abs() >= 3) | (ano['spread_bps_z'].abs() >= 3) | (ano['depth_ratio_z'].abs() >= 3)]
        if ano.empty:
            continue
        ano['run_id'] = (ano['timestamp'].diff().gt(pd.Timedelta('5min')) | ano['timestamp'].diff().isna()).cumsum()
        for _, run in ano.groupby('run_id'):
            if len(run) < 2:
                continue
            score = run[['obi_z', 'spread_bps_z', 'depth_ratio_z']].abs().max().max() + len(run)
            alerts.append({
                'sec_id': sec_id,
                'trade_date': run['trade_date'].iloc[0],
                'timestamp': run['timestamp'].min(),
                'time_window_start': run['timestamp'].min().strftime('%H:%M:%S'),
                'anomaly_type': 'multi_feature_outlier_window',
                'severity': 'MEDIUM',
                'score': round(score, 3),
                'remarks': (
                    f'DBSCAN isolated a {len(run)}-minute outlier window where multiple order-book features '
                    f'deviated from the ticker baseline at the same time. This row is kept only because at least '
                    f'one core feature also crossed a hard threshold.'
                )
            })
    return alerts


def compress_alerts(alerts: list[dict], top_n: int = 24) -> pd.DataFrame:
    if not alerts:
        return pd.DataFrame(columns=['alert_id', 'sec_id', 'trade_date', 'time_window_start', 'anomaly_type', 'severity', 'remarks', 'time_to_run'])
    df = pd.DataFrame(alerts).copy()
    df['severity_rank'] = df['severity'].map(SEVERITY_ORDER)
    df = df.sort_values(['sec_id', 'trade_date', 'timestamp', 'score', 'severity_rank'], ascending=[True, True, True, False, False]).reset_index(drop=True)

    # merge near-duplicate episodes across anomaly types inside same security/day
    kept = []
    for _, row in df.iterrows():
        ts = row['timestamp']
        duplicate = False
        for k in kept:
            if row['sec_id'] == k['sec_id'] and row['trade_date'] == k['trade_date'] and abs(ts - k['timestamp']) <= pd.Timedelta('12min'):
                duplicate = True
                if (row['score'], row['severity_rank']) > (k['score'], k['severity_rank']):
                    k.update(row.to_dict())
                break
        if not duplicate:
            kept.append(row.to_dict())

    out = pd.DataFrame(kept)
    # keep strongest episodes overall, while allowing coverage across securities
    out = out.sort_values(['severity_rank', 'score'], ascending=[False, False])
    out['rank_within_sec'] = out.groupby('sec_id').cumcount() + 1
    out = out[(out['rank_within_sec'] <= 2)]
    out = out.head(top_n).copy()
    out = out[~((out['anomaly_type'] == 'spread_depth_dislocation') & (out['time_window_start'] == '09:00:00'))].copy()
    out = out.head(top_n).sort_values(['trade_date', 'timestamp', 'sec_id']).reset_index(drop=True)
    out.insert(0, 'alert_id', range(1, len(out) + 1))
    return out[['alert_id', 'sec_id', 'trade_date', 'time_window_start', 'anomaly_type', 'severity', 'remarks']]


def run(data_dir: str | None = None, output_file: str = OUTPUT_FILE, top_n: int = 24, verbose: bool = True):
    t0 = time.time()
    if verbose:
        print('=' * 78)
        print('Problem Statement 1 - Order Book Concentration')
        print('=' * 78)
        print(f'[START] Pipeline started at: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}')

    t_load = time.time()
    base, mkt, ohlcv, trades = load_data(data_dir)
    load_elapsed = time.time() - t_load
    if verbose:
        print(f'[LOAD] Data directory: {base}')
        print(f'[LOAD] market_data rows={len(mkt):,}, ohlcv rows={len(ohlcv):,}, trade_data rows={len(trades):,}')
        print(f'[LOAD] Load time: {load_elapsed:.2f} sec')

    t_feat = time.time()
    mkt_feat = compute_market_features(mkt)
    trade_feat = compute_trade_features(trades)
    feat_elapsed = time.time() - t_feat
    if verbose:
        print(f'[FEATURES] market feature rows={len(mkt_feat):,}, trade feature rows={len(trade_feat):,}')
        print(f'[FEATURES] Feature engineering time: {feat_elapsed:.2f} sec')

    alerts = []

    t = time.time()
    imbalance_alerts = detect_imbalance_runs(mkt_feat)
    alerts += imbalance_alerts
    if verbose:
        print(f'[DETECT] sustained imbalance candidates: {len(imbalance_alerts):,} | stage time: {time.time() - t:.2f} sec')

    t = time.time()
    spread_depth_alerts = detect_spread_depth_episodes(mkt_feat)
    alerts += spread_depth_alerts
    if verbose:
        print(f'[DETECT] spread/depth dislocation candidates: {len(spread_depth_alerts):,} | stage time: {time.time() - t:.2f} sec')

    t = time.time()
    cancel_alerts = detect_cancel_bursts(trade_feat, mkt_feat)
    alerts += cancel_alerts
    if verbose:
        print(f'[DETECT] cancel burst candidates: {len(cancel_alerts):,} | stage time: {time.time() - t:.2f} sec')

    t = time.time()
    close_alerts = detect_marking_close(trade_feat, ohlcv)
    alerts += close_alerts
    if verbose:
        print(f'[DETECT] marking-the-close candidates: {len(close_alerts):,} | stage time: {time.time() - t:.2f} sec')

    t = time.time()
    dbscan_alerts = detect_dbscan_windows(mkt_feat)
    alerts += dbscan_alerts
    if verbose:
        print(f'[DETECT] DBSCAN outlier candidates: {len(dbscan_alerts):,} | stage time: {time.time() - t:.2f} sec')
        print(f'[DETECT] total raw candidate alerts before compression: {len(alerts):,}')

    t = time.time()
    out = compress_alerts(alerts, top_n=top_n)
    compress_elapsed = time.time() - t
    elapsed = round(time.time() - t0, 2)
    out['time_to_run'] = elapsed
    out.to_csv(output_file, index=False)

    if verbose:
        print(f'[COMPRESS] Compression time: {compress_elapsed:.2f} sec')
        print(f'[FINAL] Final alert rows written: {len(out):,}')
        print(f'[FINAL] Output file: {output_file}')
        print(f'[FINAL] Total runtime: {elapsed:.2f} sec')
        print('=' * 78)
    return out

if __name__ == '__main__':
    df = run()
    print(df.head(10).to_string(index=False))
    print(f'\nRows: {len(df)}')


Problem Statement 1 - Order Book Concentration
[START] Pipeline started at: 2026-03-29 02:15:31
[LOAD] Data directory: data_dir\equity
[LOAD] market_data rows=44,607, ohlcv rows=532, trade_data rows=3,036
[LOAD] Load time: 0.26 sec
[FEATURES] market feature rows=44,607, trade feature rows=3,036
[FEATURES] Feature engineering time: 1.04 sec
[DETECT] sustained imbalance candidates: 47 | stage time: 0.09 sec
[DETECT] spread/depth dislocation candidates: 78 | stage time: 0.11 sec
[DETECT] cancel burst candidates: 0 | stage time: 0.02 sec
[DETECT] marking-the-close candidates: 2 | stage time: 0.02 sec
[DETECT] DBSCAN outlier candidates: 279 | stage time: 1.60 sec
[DETECT] total raw candidate alerts before compression: 406
[COMPRESS] Compression time: 0.36 sec
[FINAL] Final alert rows written: 12
[FINAL] Output file: p1_alerts.csv
[FINAL] Total runtime: 3.50 sec
 alert_id  sec_id trade_date time_window_start                   anomaly_type severity                                             